# 01 — Data Preparation for VerifiedRAG

**Project:** VerifiedRAG — Multi-Agent RAG with Claim-Level Fact Verification

This notebook prepares a reproducible corpus of AI/ML research papers for the downstream RAG experiments.

### Pipeline

```text
Research-paper PDFs
        ↓
PDF text extraction
        ↓
Page-aware cleaning
        ↓
Recursive chunking
        ↓
Metadata attachment
        ↓
Deduplication
        ↓
Sentence-transformer embeddings
        ↓
FAISS vector index
        ↓
Saved dataset + index
```

### Outputs

The notebook creates:

```text
data/
├── raw_pdfs/
├── processed/
│   ├── chunks.jsonl
│   ├── corpus_metadata.json
│   └── papers.json
└── vector_store/
    ├── faiss.index
    ├── chunks_metadata.json
    └── index_metadata.json
```

The important design choice is that **every chunk keeps its paper title, paper ID, page number, and chunk ID**. This will later allow the verifier to cite and inspect the exact source evidence.

## 1. Install dependencies

This notebook is designed for Google Colab. It does not require a GPU for PDF processing or FAISS indexing, although a GPU can accelerate embedding generation.

In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu tqdm pandas numpy requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.2 MB/s eta 0:00:00


## 2. Imports and project configuration

In [2]:
from pathlib import Path
import json
import re
import hashlib
import requests
import numpy as np
import pandas as pd
import fitz  # PyMuPDF

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

import faiss
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
# -------------------------------------------------------------------
# Project paths
# -------------------------------------------------------------------

PROJECT_ROOT = Path("/content/Verified-RAG")

RAW_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VECTOR_DIR = PROJECT_ROOT / "data" / "vector_store"

for directory in [RAW_DIR, PROCESSED_DIR, VECTOR_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw PDFs:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Vector store:", VECTOR_DIR)


Project root: /content/Verified-RAG
Raw PDFs: /content/Verified-RAG/data/raw_pdfs
Processed data: /content/Verified-RAG/data/processed
Vector store: /content/Verified-RAG/data/vector_store


## 3. Define the research-paper corpus

For the first version, we use a small set of well-known ML papers. This is intentionally small so the complete pipeline can be tested quickly.

You can later expand the corpus by adding more PDF URLs or uploading your own papers.

**Initial corpus:**

- Attention Is All You Need
- Deep Residual Learning for Image Recognition
- BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
- Adam: A Method for Stochastic Optimization
- Language Models are Unsupervised Multitask Learners

The `paper_id` is kept separately from the filename so that changing filenames does not break downstream metadata.

In [4]:
PAPERS = [
    {
        "paper_id": "attention_2017",
        "title": "Attention Is All You Need",
        "authors": "Vaswani et al.",
        "arxiv_id": "1706.03762",
        "url": "https://arxiv.org/pdf/1706.03762.pdf",
    },
    {
        "paper_id": "resnet_2015",
        "title": "Deep Residual Learning for Image Recognition",
        "authors": "He et al.",
        "arxiv_id": "1512.03385",
        "url": "https://arxiv.org/pdf/1512.03385.pdf",
    },
    {
        "paper_id": "bert_2018",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": "Devlin et al.",
        "arxiv_id": "1810.04805",
        "url": "https://arxiv.org/pdf/1810.04805.pdf",
    },
    {
        "paper_id": "adam_2014",
        "title": "Adam: A Method for Stochastic Optimization",
        "authors": "Kingma and Ba",
        "arxiv_id": "1412.6980",
        "url": "https://arxiv.org/pdf/1412.6980.pdf",
    },
    {
        "paper_id": "gpt2_2019",
        "title": "Language Models are Unsupervised Multitask Learners",
        "authors": "Radford et al.",
        "arxiv_id": "not_used",
        "url": "https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf",
    },
]

pd.DataFrame(PAPERS)[["paper_id", "title", "authors", "url"]]


,paper_id,title,authors,url
0,attention_2017,Attention Is All You Need,Vaswani et al.,https://arxiv.org/pdf/1706.03762.pdf
1,resnet_2015,Deep Residual Learning for Image Recognition,He et al.,https://arxiv.org/pdf/1512.03385.pdf
2,bert_2018,BERT: Pre-training of Deep Bidirectional Trans...,Devlin et al.,https://arxiv.org/pdf/1810.04805.pdf
3,adam_2014,Adam: A Method for Stochastic Optimization,Kingma and Ba,https://arxiv.org/pdf/1412.6980.pdf
4,gpt2_2019,Language Models are Unsupervised Multitask Lea...,Radford et al.,https://cdn.openai.com/better-language-models/...


## 4. Download PDFs

The downloader skips files that already exist. This makes the notebook safe to re-run.

If a download fails, the notebook reports the error instead of silently creating an incomplete corpus.

In [5]:
def download_file(url: str, destination: Path, timeout: int = 60):
    response = requests.get(
        url,
        timeout=timeout,
        headers={"User-Agent": "VerifiedRAG/1.0 research project"}
    )
    response.raise_for_status()

    content_type = response.headers.get("content-type", "").lower()
    if "pdf" not in content_type and not response.content.startswith(b"%PDF"):
        raise ValueError(
            f"Downloaded content does not appear to be a PDF. "
            f"Content-Type={content_type}"
        )

    destination.write_bytes(response.content)


download_results = []

for paper in PAPERS:
    destination = RAW_DIR / f"{paper['paper_id']}.pdf"

    if destination.exists() and destination.stat().st_size > 10_000:
        status = "already_exists"
    else:
        try:
            download_file(paper["url"], destination)
            status = "downloaded"
        except Exception as exc:
            status = f"ERROR: {exc}"

    download_results.append({
        "paper_id": paper["paper_id"],
        "title": paper["title"],
        "path": str(destination),
        "status": status,
        "size_bytes": destination.stat().st_size if destination.exists() else 0,
    })

download_df = pd.DataFrame(download_results)
display(download_df)


,paper_id,title,path,status,size_bytes
0,attention_2017,Attention Is All You Need,/content/Verified-RAG/data/raw_pdfs/attention_...,downloaded,2215244
1,resnet_2015,Deep Residual Learning for Image Recognition,/content/Verified-RAG/data/raw_pdfs/resnet_201...,downloaded,819383
2,bert_2018,BERT: Pre-training of Deep Bidirectional Trans...,/content/Verified-RAG/data/raw_pdfs/bert_2018.pdf,downloaded,775166
3,adam_2014,Adam: A Method for Stochastic Optimization,/content/Verified-RAG/data/raw_pdfs/adam_2014.pdf,downloaded,584641
4,gpt2_2019,Language Models are Unsupervised Multitask Lea...,/content/Verified-RAG/data/raw_pdfs/gpt2_2019.pdf,downloaded,582775


## 5. Inspect PDF files

In [6]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDFs found in {RAW_DIR}. "
        "Check the download step or upload PDFs manually."
    )

for path in pdf_files:
    print(f"{path.name:45s} {path.stat().st_size / 1024:.1f} KB")


adam_2014.pdf                                 570.9 KB
attention_2017.pdf                            2163.3 KB
bert_2018.pdf                                 757.0 KB
gpt2_2019.pdf                                 569.1 KB
resnet_2015.pdf                               800.2 KB


## 6. Extract page-level text

We preserve page boundaries because the verifier will eventually need to answer questions such as:

> Which exact source passage supports this claim?

Each extracted page becomes a document object containing the paper ID, page number, and raw text.

In [7]:
def normalize_whitespace(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\u00ad", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_pdf_pages(pdf_path: Path, paper_id: str, title: str):
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = normalize_whitespace(text)

            if not text:
                continue

            pages.append({
                "paper_id": paper_id,
                "title": title,
                "page": page_number,
                "text": text,
            })

    return pages


paper_lookup = {paper["paper_id"]: paper for paper in PAPERS}

all_pages = []

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    paper_id = pdf_path.stem
    paper = paper_lookup.get(paper_id)

    if paper is None:
        print(f"Skipping unknown PDF: {pdf_path.name}")
        continue

    all_pages.extend(
        extract_pdf_pages(
            pdf_path,
            paper_id=paper_id,
            title=paper["title"],
        )
    )

print(f"Extracted pages: {len(all_pages)}")


Extracting PDFs:   0%|          | 0/5 [00:00<?, ?it/s]

Extracted pages: 82


In [8]:
pages_df = pd.DataFrame(all_pages)

if pages_df.empty:
    raise ValueError(
        "No text was extracted. Some PDFs may be scanned/image-only "
        "and require OCR."
    )

display(
    pages_df.groupby(["paper_id", "title"], as_index=False)
    .agg(pages=("page", "count"))
)

print("\nSample page:")
print(pages_df.iloc[0]["text"][:2000])


,paper_id,title,pages
0,adam_2014,Adam: A Method for Stochastic Optimization,15
1,attention_2017,Attention Is All You Need,15
2,bert_2018,BERT: Pre-training of Deep Bidirectional Trans...,16
3,gpt2_2019,Language Models are Unsupervised Multitask Lea...,24
4,resnet_2015,Deep Residual Learning for Image Recognition,12



Sample page:
Published as a conference paper at ICLR 2015
ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION
Diederik P. Kingma*
University of Amsterdam, OpenAI
dpkingma@openai.com
Jimmy Lei Ba∗
University of Toronto
jimmy@psi.utoronto.ca
ABSTRACT
We introduce Adam, an algorithm for ﬁrst-order gradient-based optimization of
stochastic objective functions, based on adaptive estimates of lower-order mo-
ments. The method is straightforward to implement, is computationally efﬁcient,
has little memory requirements, is invariant to diagonal rescaling of the gradients,
and is well suited for problems that are large in terms of data and/or parameters.
The method is also appropriate for non-stationary objectives and problems with
very noisy and/or sparse gradients. The hyper-parameters have intuitive interpre-
tations and typically require little tuning. Some connections to related algorithms,
on which Adam was inspired, are discussed. We also analyze the theoretical con-
vergence properties of the a

## 7. Page-level cleaning

We apply conservative cleaning.

We **do not** aggressively remove punctuation, mathematical symbols, or technical terms because those can be important evidence in scientific documents.

We only remove obvious extraction noise and repeated whitespace.

In [9]:
def clean_page_text(text: str) -> str:
    text = text.replace("-\n", "")
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for page in all_pages:
    page["clean_text"] = clean_page_text(page["text"])

all_pages = [
    page for page in all_pages
    if len(page["clean_text"]) >= 80
]

print(f"Usable pages after cleaning: {len(all_pages)}")


Usable pages after cleaning: 82


## 8. Recursive text chunking

A fixed-size chunker is simple but can split scientific explanations at awkward points.

For this first reproducible version, we use a lightweight recursive splitter with:

- target chunk size: 1,000 characters
- overlap: 150 characters
- paragraph/sentence/word boundaries when possible

The exact chunking parameters will later be treated as experimental hyperparameters.

In [10]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150


def split_text_recursive(text: str, chunk_size=1000, overlap=150):
    separators = ["\n\n", ". ", "; ", ", ", " "]

    def recursive_split(current_text, separator_index=0):
        if len(current_text) <= chunk_size:
            return [current_text.strip()] if current_text.strip() else []

        if separator_index >= len(separators):
            # Last resort: hard split.
            chunks = []
            start = 0
            while start < len(current_text):
                end = min(start + chunk_size, len(current_text))
                piece = current_text[start:end].strip()
                if piece:
                    chunks.append(piece)
                start = end - overlap if end < len(current_text) else end
            return chunks

        separator = separators[separator_index]
        parts = current_text.split(separator)

        if len(parts) == 1:
            return recursive_split(current_text, separator_index + 1)

        groups = []
        current = ""

        for part in parts:
            candidate = (
                part if not current
                else current + separator + part
            )

            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current.strip():
                    groups.append(current.strip())
                current = part

        if current.strip():
            groups.append(current.strip())

        # Apply overlap between adjacent groups.
        merged = []
        for group in groups:
            if not merged:
                merged.append(group)
                continue

            previous = merged[-1]
            overlap_text = previous[-overlap:] if overlap > 0 else ""
            candidate = (overlap_text + " " + group).strip()

            if len(candidate) <= chunk_size + overlap:
                merged.append(candidate)
            else:
                merged.append(group)

        # Recursively split anything still too large.
        final = []
        for group in merged:
            if len(group) <= chunk_size:
                final.append(group)
            else:
                final.extend(
                    recursive_split(group, separator_index + 1)
                )

        return final

    return recursive_split(text)


chunks = []

for page in tqdm(all_pages, desc="Chunking pages"):
    page_chunks = split_text_recursive(
        page["clean_text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP,
    )

    for local_index, chunk_text in enumerate(page_chunks):
        chunks.append({
            "paper_id": page["paper_id"],
            "title": page["title"],
            "page": page["page"],
            "chunk_on_page": local_index,
            "text": chunk_text,
        })

print("Total chunks:", len(chunks))


Chunking pages:   0%|          | 0/82 [00:00<?, ?it/s]

Total chunks: 516


## 9. Deduplicate chunks

In [11]:
def text_hash(text: str) -> str:
    normalized = re.sub(r"\s+", " ", text.strip().lower())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


seen = set()
deduplicated_chunks = []

for chunk in chunks:
    key = text_hash(chunk["text"])

    if key in seen:
        continue

    seen.add(key)
    chunk["text_hash"] = key
    deduplicated_chunks.append(chunk)

print("Before deduplication:", len(chunks))
print("After deduplication: ", len(deduplicated_chunks))
print("Removed:             ", len(chunks) - len(deduplicated_chunks))


Before deduplication: 516
After deduplication:  516
Removed:              0


## 10. Assign stable chunk IDs

Stable IDs are essential for reproducible citations and verification.

A verifier can later return something like:

```text
paper_id = resnet_2015
page = 4
chunk_id = resnet_2015_p004_c002
```

rather than an opaque vector-database ID.

In [12]:
for global_index, chunk in enumerate(deduplicated_chunks):
    chunk["chunk_index"] = global_index
    chunk["chunk_id"] = (
        f"{chunk['paper_id']}"
        f"_p{chunk['page']:03d}"
        f"_c{chunk['chunk_on_page']:03d}"
    )

chunks = deduplicated_chunks

print("Example chunk metadata:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:2500])


Example chunk metadata:
{
  "paper_id": "adam_2014",
  "title": "Adam: A Method for Stochastic Optimization",
  "page": 1,
  "chunk_on_page": 0,
  "text": "Published as a conference paper at ICLR 2015 ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION Diederik P. Kingma* University of Amsterdam, OpenAI dpkingma@openai.com Jimmy Lei Ba∗ University of Toronto jimmy@psi.utoronto.ca ABSTRACT We introduce Adam, an algorithm for ﬁrst-order gradient-based optimization of stochastic objective functions, based on adaptive estimates of lower-order moments. The method is straightforward to implement, is computationally efﬁcient, has little memory requirements, is invariant to diagonal rescaling of the gradients, and is well suited for problems that are large in terms of data and/or parameters. The method is also appropriate for non-stationary objectives and problems with very noisy and/or sparse gradients. The hyper-parameters have intuitive interpretations and typically require little tuning. Some conne

## 11. Inspect chunk statistics

In [13]:
chunk_lengths = np.array([len(chunk["text"]) for chunk in chunks])

print("Number of chunks:", len(chunk_lengths))
print("Minimum length: ", chunk_lengths.min())
print("Mean length:    ", round(chunk_lengths.mean(), 2))
print("Median length:  ", round(np.median(chunk_lengths), 2))
print("Maximum length: ", chunk_lengths.max())

stats_df = (
    pd.DataFrame(chunks)
    .groupby(["paper_id", "title"], as_index=False)
    .agg(
        chunks=("chunk_id", "count"),
        avg_chars=("text", lambda x: round(x.str.len().mean(), 1)),
        min_chars=("text", lambda x: x.str.len().min()),
        max_chars=("text", lambda x: x.str.len().max()),
    )
)

display(stats_df)


Number of chunks: 516
Minimum length:  55
Mean length:     700.12
Median length:   822.5
Maximum length:  1000


,paper_id,title,chunks,avg_chars,min_chars,max_chars
0,adam_2014,Adam: A Method for Stochastic Optimization,70,714.3,216,999
1,attention_2017,Attention Is All You Need,68,695.9,204,998
2,bert_2018,BERT: Pre-training of Deep Bidirectional Trans...,106,722.4,190,1000
3,gpt2_2019,Language Models are Unsupervised Multitask Lea...,166,692.2,156,1000
4,resnet_2015,Deep Residual Learning for Image Recognition,106,683.5,55,1000


## 12. Save processed chunks

We use JSON Lines (`.jsonl`) because it is easy to inspect, stream, and process without loading the entire corpus into memory.

In [14]:
chunks_path = PROCESSED_DIR / "chunks.jsonl"

with chunks_path.open("w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

papers_path = PROCESSED_DIR / "papers.json"

with papers_path.open("w", encoding="utf-8") as f:
    json.dump(PAPERS, f, indent=2, ensure_ascii=False)

corpus_metadata = {
    "num_papers": len(PAPERS),
    "num_pages": len(all_pages),
    "num_chunks": len(chunks),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
}

metadata_path = PROCESSED_DIR / "corpus_metadata.json"

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(corpus_metadata, f, indent=2)

print("Saved:")
print(chunks_path)
print(papers_path)
print(metadata_path)


Saved:
/content/Verified-RAG/data/processed/chunks.jsonl
/content/Verified-RAG/data/processed/papers.json
/content/Verified-RAG/data/processed/corpus_metadata.json


## 13. Load the embedding model

`all-MiniLM-L6-v2` produces 384-dimensional sentence embeddings.

We normalize embeddings so that FAISS inner-product search corresponds to cosine similarity.

In [15]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device
)

print("Model:", EMBEDDING_MODEL_NAME)
print("Device:", device)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda
Embedding dimension: 384


/tmp/ipykernel_2320/335280999.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


## 14. Generate embeddings

For a larger corpus, this may take some time. On Colab, the GPU will be used automatically if available.

In [16]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embeddings = embeddings.astype("float32")

print("Embedding matrix shape:", embeddings.shape)


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding matrix shape: (516, 384)


## 15. Build the FAISS index

In [17]:
dimension = embeddings.shape[1]

# Inner product + normalized vectors = cosine similarity.
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index type:", type(index).__name__)
print("Vectors in index:", index.ntotal)
print("Vector dimension:", index.d)


FAISS index type: IndexFlatIP
Vectors in index: 516
Vector dimension: 384


## 16. Save the vector store

FAISS stores the vectors, while `chunks_metadata.json` maps vector positions back to human-readable evidence.

This separation is deliberate: vector search and source evidence remain independently inspectable.

In [18]:
faiss_path = VECTOR_DIR / "faiss.index"
metadata_path = VECTOR_DIR / "chunks_metadata.json"
index_metadata_path = VECTOR_DIR / "index_metadata.json"

faiss.write_index(index, str(faiss_path))

metadata_records = [
    {
        "vector_index": i,
        "chunk_id": chunk["chunk_id"],
        "paper_id": chunk["paper_id"],
        "title": chunk["title"],
        "page": chunk["page"],
        "chunk_on_page": chunk["chunk_on_page"],
        "text": chunk["text"],
    }
    for i, chunk in enumerate(chunks)
]

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata_records, f, indent=2, ensure_ascii=False)

index_metadata = {
    "index_type": "IndexFlatIP",
    "metric": "inner_product",
    "vectors_normalized": True,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "dimension": dimension,
    "num_vectors": len(chunks),
}

with index_metadata_path.open("w", encoding="utf-8") as f:
    json.dump(index_metadata, f, indent=2)

print("Saved:")
print(faiss_path)
print(metadata_path)
print(index_metadata_path)


Saved:
/content/Verified-RAG/data/vector_store/faiss.index
/content/Verified-RAG/data/vector_store/chunks_metadata.json
/content/Verified-RAG/data/vector_store/index_metadata.json


## 17. Test retrieval end-to-end

Before moving to the baseline RAG notebook, we verify that the persisted vector store can retrieve meaningful evidence.

This is intentionally a simple retrieval test. The actual Retriever Agent will be implemented later.

In [19]:
def load_vector_store():
    loaded_index = faiss.read_index(str(VECTOR_DIR / "faiss.index"))

    with (VECTOR_DIR / "chunks_metadata.json").open(
        "r", encoding="utf-8"
    ) as f:
        loaded_metadata = json.load(f)

    return loaded_index, loaded_metadata


loaded_index, loaded_metadata = load_vector_store()

def retrieve(query: str, top_k: int = 5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = loaded_index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        record = loaded_metadata[int(idx)].copy()
        record["score"] = float(score)
        results.append(record)

    return results


query = "What problem does residual learning solve in deep neural networks?"

results = retrieve(query, top_k=5)

for rank, result in enumerate(results, start=1):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Paper: {result['title']}")
    print(f"Page: {result['page']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(result["text"][:1200])


Rank: 1
Score: 0.6450
Paper: Deep Residual Learning for Image Recognition
Page: 1
Chunk ID: resnet_2015_p001_c000
Deep Residual Learning for Image Recognition Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun Microsoft Research {kahe, v-xiangz, v-shren, jiansun}@microsoft.com Abstract Deeper neural networks are more difﬁcult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to optimize, and can gain accuracy from considerably increased depth. On the ImageNet dataset we evaluate residual nets with a depth of up to 152 layers—8× deeper than VGG nets [41] but still having lower complexity. An ensemble of these residual nets achieves 3.57% error on the ImageNet test 

## 18. Quality checks

These checks make the data-preparation stage fail loudly rather than allowing bad data to propagate into later experiments.

In [20]:
# Basic integrity checks

assert len(PAPERS) > 0, "No papers configured."
assert len(chunks) > 0, "No chunks created."
assert embeddings.shape[0] == len(chunks), "Embedding/chunk count mismatch."
assert loaded_index.ntotal == len(chunks), "FAISS/chunk count mismatch."
assert len(loaded_metadata) == len(chunks), "Metadata/chunk count mismatch."

chunk_ids = [chunk["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate chunk IDs found."

assert all(
    isinstance(chunk["text"], str) and len(chunk["text"]) > 0
    for chunk in chunks
), "Empty chunk detected."

print("All integrity checks passed.")


All integrity checks passed.


## 19. Final dataset summary

This summary is saved so later notebooks can verify exactly which corpus and preprocessing configuration produced the experiments.

In [21]:
final_summary = {
    "project": "VerifiedRAG",
    "stage": "01_data_preparation",
    "num_source_papers": len(PAPERS),
    "num_extracted_pages": len(all_pages),
    "num_final_chunks": len(chunks),
    "chunk_size_chars": CHUNK_SIZE,
    "chunk_overlap_chars": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(dimension),
    "faiss_index": "IndexFlatIP",
    "normalized_embeddings": True,
    "device_used_for_embeddings": device,
}

print(json.dumps(final_summary, indent=2))

with (PROCESSED_DIR / "preparation_summary.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(final_summary, f, indent=2)

print("\nData preparation complete.")


{
  "project": "VerifiedRAG",
  "stage": "01_data_preparation",
  "num_source_papers": 5,
  "num_extracted_pages": 82,
  "num_final_chunks": 516,
  "chunk_size_chars": 1000,
  "chunk_overlap_chars": 150,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dimension": 384,
  "faiss_index": "IndexFlatIP",
  "normalized_embeddings": true,
  "device_used_for_embeddings": "cuda"
}

Data preparation complete.


In [ ]:
import os
import requests
import json
from pathlib import Path
from google.colab import _message

# 1. Download current in-memory notebook state to notebooks/01_data_preparation.ipynb
notebook_json = _message.blocking_request('get_ipynb')
target_nb_path = Path("/content/Verified-RAG/notebooks/01_data_preparation.ipynb")

with open(target_nb_path, "w", encoding="utf-8") as f:
    json.dump(notebook_json["ipynb"], f, indent=2)

print(f"Saved notebook to {target_nb_path}")